# Retinal Vessel Segmentation — Training on Google Colab (GPU)

Trains the U-Net (+ rotation head) on **DRIVE** with the corrected pipeline
(data augmentation now actually applied during training).

**Before running:** `Runtime > Change runtime type > Hardware accelerator = GPU`.

**Steps:** upload `colab_package.zip` → install deps → train → plot curves → download `best_model.pth`.

Build the zip locally first with: `python build_colab_package.py --train-only --no-checkpoint`

In [ ]:
# Cell 1: Upload colab_package.zip and extract
from google.colab import files
import zipfile, os, sys

print('Select colab_package.zip when the file picker opens...')
uploaded = files.upload()   # pick colab_package.zip

zip_name = list(uploaded.keys())[0]
print(f'Extracting {zip_name} ...')
with zipfile.ZipFile(zip_name, 'r') as z:
    for member in z.infolist():
        member.filename = member.filename.replace('\\', '/')  # fix Windows paths
        z.extract(member, '/content/')

PROJECT_PATH = '/content/colab_package'
assert os.path.isdir(PROJECT_PATH), f'Extraction failed. Contents: {os.listdir("/content/")}'
sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)
print('Ready. Working directory:', os.getcwd())

In [ ]:
# Cell 2: Install dependencies (Colab already has torch + GPU build)
!pip install -q scikit-learn Pillow tqdm matplotlib

In [ ]:
# Cell 3: Verify GPU and imports
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Set Runtime > Change runtime type > GPU, then rerun.')

from models.unet import UNetWithRotationHead
from training.dataset import get_drive_loaders
from training.train import train_one_epoch, validate, CombinedLoss
print('Imports OK')

In [ ]:
# Cell 4: Confirm DRIVE data is present and inspect the train/val split
from pathlib import Path
n_img = len(list(Path('data/DRIVE/images').glob('*')))
n_msk = len(list(Path('data/DRIVE/masks').glob('*')))
print(f'DRIVE images: {n_img} | masks: {n_msk}')
assert n_img > 0 and n_img == n_msk, 'DRIVE data missing or mismatched.'

tl, vl = get_drive_loaders('data/DRIVE', img_size=512, batch_size=4)
print(f'Train batches: {len(tl)} | Val batches: {len(vl)}')
print('(Augmentation is ON for train, OFF for val — the fixed split.)')

In [ ]:
# Cell 5: Train
import time, json, os
import torch.nn as nn, torch.optim as optim

EPOCHS     = 60
BATCH_SIZE = 4
LR         = 1e-4
LAMBDA_ROT = 0.3
IMG_SIZE   = 512

train_loader, val_loader = get_drive_loaders('data/DRIVE', img_size=IMG_SIZE, batch_size=BATCH_SIZE)

model     = UNetWithRotationHead(n_channels=3, n_classes=1).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
seg_crit  = CombinedLoss().to(device)
rot_crit  = nn.CrossEntropyLoss().to(device)

os.makedirs('checkpoints', exist_ok=True)
history = {'train_loss': [], 'val_loss': [], 'val_dice': []}
best_dice = 0.0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, seg_crit, rot_crit,
                                 optimizer, device, lambda_rot=LAMBDA_ROT)
    val_loss, val_dice = validate(model, val_loader, seg_crit, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)

    flag = ''
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_dice': val_dice},
                   'checkpoints/best_model.pth')
        flag = '  *saved best*'

    print(f'Epoch {epoch:3d}/{EPOCHS}  train_loss={train_loss:.4f}  '
          f'val_loss={val_loss:.4f}  val_dice={val_dice:.4f}  ({time.time()-t0:.1f}s){flag}')

json.dump(history, open('checkpoints/history.json', 'w'))
print(f'\nDone. Best val Dice = {best_dice:.4f}  ->  checkpoints/best_model.pth')

In [ ]:
# Cell 6: Plot training curves
import json
import matplotlib.pyplot as plt
hist = json.load(open('checkpoints/history.json'))
epochs = range(1, len(hist['val_dice']) + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, hist['train_loss'], label='train')
ax[0].plot(epochs, hist['val_loss'],   label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(epochs, hist['val_dice'], color='green')
ax[1].set_title('Validation Dice'); ax[1].set_xlabel('epoch'); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=120)
plt.show()
print('Best val Dice:', max(hist['val_dice']))

In [ ]:
# Cell 7: Download the trained checkpoint and artifacts to your PC
from google.colab import files
files.download('checkpoints/best_model.pth')
files.download('checkpoints/history.json')
files.download('checkpoints/training_curves.png')
print('Place best_model.pth into your local checkpoints/ folder, then run evaluation.')